# W9 作業｜解釋記憶門診的預測模型（SHAP）
**智慧醫療與機器學習基礎**　臺北醫學大學 智慧醫療學士學位學程　授課教師：王三源　課程助教：古珉瑄（醫資所，m610115009@tmu.edu.tw）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wang3yuan/FIHML/blob/main/homework/W09_HW.ipynb)

**開始前**：先「檔案 → 在雲端硬碟中儲存複本」，在自己的複本上作答。
**繳交**：完成後「檔案 → 下載 → 下載 .ipynb」，上傳至 Im@TMU。**期限：11/15（日）23:59**
**檔名**：`W09_HW_學號_姓名.ipynb`

### 情境
延續 W7 作業：你是記憶門診的資料分析助理。門診已經採用隨機森林，幫忙篩出「需要進一步做完整失智評估」的病人。門診主任現在問兩個問題：
> 「整體來說，模型是看哪些資料做判斷的？」
> 「為什麼這位病人被判成高風險（或低風險）？」

請用課堂學過的 **SHAP** 回答，並用主任聽得懂的話說明。

### 資料
延續 W5–W7 的 Kaggle [Alzheimer's Disease Dataset](https://www.kaggle.com/datasets/rabieelkharoua/alzheimers-disease-dataset)（Rabie El Kharoua, 2024，DOI: [10.34740/KAGGLE/DSV/8668279](https://doi.org/10.34740/KAGGLE/DSV/8668279)，CC BY 4.0；作者註明為**教學用合成資料**），使用清理好的 `alzheimers_clean.csv`（2,149 位病人）。欄位說明請見 W7 作業。`Diagnosis` 1 = 阿茲海默症（AD）、0 = 非 AD。

### 執行環境
本作業已在 **scikit-learn 1.6.1／pandas 2.2.3** 與 **scikit-learn 1.9.1／pandas 3.0.6** 測試通過，兩組都搭配 **shap 0.52.0**（第一格會自動安裝並印出版本）。查手冊或問 Gemini 時，請說明你用的是 **shap 0.52.0**。

### 評分標準（100 分）
| 項目 | 配分 | 評分重點 |
|---|---|---|
| 1. 準備資料與模型 | 10 | 與 W7 作業相同的切分、隨機森林，正確率 |
| 2. 算 SHAP 值 | 15 | 解釋器、取出 AD 那一類、驗證基準值＋SHAP 值加總 |
| 3. 整體解釋 | 20 | 長條圖、蜂群圖、前 3 名與方向、和特徵重要性比較 |
| 4. 解釋個別病人 | 20 | 兩位病人的瀑布圖、用文字說出原因 |
| 5. 依賴圖 | 15 | 兩個特徵的依賴圖與形狀解讀 |
| 6. 向主任報告與限制 | 20 | 白話說明、指出限制（相關 ≠ 因果等） |
| GenAI 使用說明 | 必填 | 如實揭露不扣分；使用了卻未揭露，作業成績上限 60 分 |

> 💡 程式寫法可以參考課堂 notebook（`notebooks/W09_shap.ipynb`）：每個小題都有對應的範例，通常只要改欄位名稱或數字。
> 💡 卡住的順序：① 錯誤訊息的最後一行 → ② Colab 的 Gemini（按「解釋錯誤」）→ ③ 課堂 notebook 的範例。Gemini 可能用舊版的 shap 寫法，要對照課堂範例查證，並在最後的「GenAI 使用說明」揭露。
> 💡 繳交前「執行階段 → 重新啟動並全部執行」，確認從頭到尾沒有錯誤。

In [ ]:
# 第一次執行會自動安裝 shap（約 1 分鐘，Colab 重新連線後要再裝一次）
try:
    import shap
except ModuleNotFoundError:
    %pip install -q shap==0.52.0
    import shap

import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import urllib.request
import matplotlib.font_manager as fm

print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| shap", shap.__version__)

# 中文字型（與課堂相同）
for name in ["NotoSansTC-Regular.ttf", "NotoSansTC-Bold.ttf"]:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/wang3yuan/FIHML/main/fonts/" + name, name)
    fm.fontManager.addfont(name)
plt.rcParams["font.family"] = "Noto Sans TC"
plt.rcParams["axes.unicode_minus"] = False

ad = pd.read_csv("https://raw.githubusercontent.com/wang3yuan/FIHML/main/data/hw/alzheimers_clean.csv")
print(ad.shape)
ad.head()

---
## 1. 準備資料與模型（10 分）
和 W7 作業相同：
- `X`：拿掉 `PatientID` 與 `Diagnosis`，`Ethnicity` 用 `pd.get_dummies(X, columns=["Ethnicity"], drop_first=True, dtype=int)` 轉成 0／1；`y`：`Diagnosis`
- `train_test_split(X, y, test_size=0.25, stratify=y, random_state=1151)`
- 訓練 `RandomForestClassifier(random_state=1151)`，存成變數 `forest`，印出訓練集與測試集的正確率

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# 參考 W7 作業與課堂 notebook：建立 X、y，切分，訓練隨機森林 forest，印出正確率
# TODO


---
## 2. 算 SHAP 值（15 分）
**(a)** 用 `shap.TreeExplainer(forest)` 建立解釋器，對**測試集**算出 SHAP 值，存成 `sv`，印出形狀。
形狀有三個數字：病人數 × 特徵數 × 類別數。

**(b)** 只留下「AD」那一類（第 1 類）：`sv1 = sv[:, :, 1]`，之後的圖都用 `sv1`。

**(c)** 驗證：拿測試集的第 0 位病人，印出「基準值 ＋ SHAP 值加總」和 `forest.predict_proba` 給的 AD 機率，兩個數字是否一樣？（參考課堂 notebook 1.3 的驗證）

In [ ]:
# (a) 建立解釋器、算 SHAP 值 sv，印出形狀
# TODO

# (b) 只留下 AD 那一類，存成 sv1
# TODO

# (c) 驗證第 0 位病人：基準值 + SHAP 值加總 = predict_proba
# TODO


**(d)** 基準值（約 0.35）的意思是什麼？SHAP 值為正、為負各代表什麼？（用一兩句話）

> 你的回答：


---
## 3. 整體解釋（20 分）
**(a)** 畫出前 10 名特徵的**長條圖**（`shap.plots.bar`）和**蜂群圖**（`shap.plots.beeswarm`），都要加上自己的標題（`show=False` 再 `plt.title`）。

In [ ]:
# 範例（課堂 notebook 1.4）：
# shap.plots.bar(sv1, max_display=10, show=False)
# plt.title("...")
# plt.show()
# TODO：畫出長條圖與蜂群圖，都加上標題


**(b)** 前 3 名是哪些特徵？看蜂群圖，每個特徵是「值越高越推向 AD」還是「值越高越拉向非 AD」？用一句話解讀。

**(c)** 和 W7 的 `forest.feature_importances_` 比較：印出兩種方法的前 8 名，哪些特徵兩邊都排前面？有沒有哪個特徵在兩邊的排名差很多（例如 `BMI`、`DietQuality`、`AlcoholConsumption`）？為什麼可能不同？（提示：W9 課堂「身高之謎」）

In [ ]:
shap_imp = pd.Series(np.abs(sv1.values).mean(axis=0), index=X.columns).sort_values(ascending=False)
# TODO：再建立特徵重要性 imp（參考 W7），印出兩種方法的前 8 名；
#       想比較某幾個特徵的排名，可以用 .rank(ascending=False)


> 你的回答：


---
## 4. 解釋個別病人（20 分）
**(a)** 找出測試集中，模型預測 AD 機率**最高**的病人（`np.argmax`），存成 `i`。畫出他的瀑布圖，列出推高最多的前 3 個特徵與特徵值，用文字說明模型為什麼認為他很可能是 AD。

In [ ]:
# 範例（課堂 notebook 2.1）：
# p = forest.predict_proba(X_test)[:, 1]
# i = int(np.argmax(p))
# shap.plots.waterfall(sv1[i], max_display=8, show=False)
# TODO：找出機率最高的病人 i，畫瀑布圖，並列出推高最多的前 3 個特徵與它們的特徵值


**(b)** 找出一位「**真的是 AD、但模型預測機率低於 0.5**」的病人（被漏掉的病人），存成 `j`（參考課堂 notebook 2.1 的 `missed`）。畫出他的瀑布圖。

**(c)** 模型為什麼漏掉這位病人？哪些特徵把預測往下拉？你覺得門診可以怎麼補救（例如多做哪項評估）？

In [ ]:
# TODO：找出被漏掉的病人 j（真的是 AD、預測機率 < 0.5），畫瀑布圖，
#       並列出 SHAP 值絕對值最大的 5 個特徵與它們的特徵值


> 你的回答：


---
## 5. 依賴圖（15 分）
**(a)** 畫出 `MMSE` 的依賴圖（`shap.plots.scatter(sv1[:, "MMSE"])`）。MMSE 越高，SHAP 值怎麼變化？SHAP 值由負轉正（或由正轉負）的轉折點大約在多少分？

**(b)** 從 `FunctionalAssessment`、`ADL`、`Age`、`SleepQuality` 裡**再選一個**，畫出依賴圖並解讀形狀（例如是一條斜線、階梯、還是幾乎平的？）。

In [ ]:
# 範例（課堂 notebook 2.2）：
# shap.plots.scatter(sv1[:, "MMSE"], show=False)
# plt.axhline(0, color="gray", linestyle="--")   # 0 的虛線，方便看正負
# plt.title("...")
# plt.show()
# TODO：畫出 MMSE 的依賴圖，再選一個特徵畫出依賴圖


> 你的回答：


---
## 6. 向主任報告與限制（20 分）
**(a)** 用 **3–4 句話**回答主任的兩個問題（用日常用語，不要用「SHAP 值」「特徵」這類術語）：
1. 整體來說，模型最看重哪些資料？（根據第 3 題）
2. 剛才預測機率最高的病人（第 4 題 (a)），為什麼被判成高風險？

**(b)** 在你的報告最後加一段**提醒（2–3 句）**：這個結果有什麼限制？請至少包含以下**兩點**，並結合這份資料說明：
- SHAP 解釋的是「模型」，顯示的是**相關**，不是**因果**
- 這份資料是**教學用合成資料**，結果不能直接用在真實病人

**(c)** 想一想：如果門診想把模型放在「**初診篩檢**」使用，下列哪個資料**不適合**當特徵？為什麼？
（甲）`MMSE`　（乙）`MemoryComplaints`　（丙）「是否已經開始服用失智症藥物」（這份資料裡沒有，假設有）

> 你的回答：


---
## GenAI 使用說明（必填，不計分）
依課程〈生成式 AI 使用指引〉：**如實揭露不會扣分**；使用了卻未揭露，作業成績上限 60 分。沒有使用也請寫「本次作業未使用 GenAI」。

> 本次作業是否使用 GenAI：
>
> 用了哪些工具（例如 Colab 的 Gemini、ChatGPT）：
>
> 用在哪一題、怎麼問（例如：第 4 題請 Gemini 用中文解釋瀑布圖怎麼讀）：
>
> 怎麼查證、有沒有修改它給的內容（shap 的寫法要對照課堂範例與版本）：